# **7일차 팀 프로젝트: 보안 분석 에이전트**

이 노트북은 현재 GitHub 프로젝트인 `security-agent-langgraph`의 내용을 기준으로 정리한 실습/발표용 문서입니다.

이 프로젝트는 한마디로 **소스코드나 설정 파일을 읽고, 보안 문제가 있는지 자동으로 1차 점검해주는 보안 분석 에이전트**입니다.

## 학습 목표
1. 기존 코딩 에이전트가 보안 분석 에이전트로 바뀐 지점을 이해합니다.
2. `agent.py`가 에이전트의 역할과 응답 원칙을 어떻게 정의하는지 확인합니다.
3. `tools.py`에 구현된 보안 도구들을 직접 실행해봅니다.
4. 민감정보 탐지, 정적 보안 점검, 체크리스트, 위험도 계산 흐름을 테스트합니다.
5. 이 프로젝트의 한계와 앞으로의 발전 방향을 설명할 수 있게 정리합니다.


## 1. 프로젝트 한 줄 설명

**LLM + 커스텀 보안 도구를 연결한 1차 보안 분석 에이전트**입니다.

사용자가 예를 들어 `tools.py 파일 보안 검사해줘`라고 요청하면 에이전트는 파일을 읽고, 민감정보 노출이나 위험한 코드 패턴을 검사한 뒤, 왜 위험한지와 어떻게 수정하면 좋은지를 한글로 정리합니다.

중요한 점은 이 프로젝트가 전문 SAST 도구를 완전히 대체하는 것이 아니라, 보안 점검을 빠르게 시작하기 위한 **1차 자동화 구조**라는 점입니다.


## 2. GitHub 업로드 파일 구조

현재 저장소는 아래 파일 구조를 기준으로 고정합니다. `.env` 파일은 API Key 같은 민감정보가 들어가므로 GitHub에 업로드하지 않습니다.

```text
Agent 팀 깃허브
├─ day7_team_project_template.ipynb
├─ agent.py
├─ tools.py
├─ langgraph.json
├─ .env (깃 업로드 X)
├─ pyproject.toml
├─ uv.lock
└─ README.md
```


In [ ]:
from pathlib import Path

project_files = [
    'day7_team_project_template.ipynb',
    'agent.py',
    'tools.py',
    'langgraph.json',
    'pyproject.toml',
    'uv.lock',
    'README.md',
]

for file_name in project_files:
    path = Path(file_name)
    status = '있음' if path.exists() else '없음'
    print(f'{file_name}: {status}')

print(f".env 업로드 여부 확인: {Path('.env').exists()} (로컬 실행용 파일이며 GitHub에는 올리지 않음)")


## 3. 전체 동작 구조

```text
사용자
  ↓
보안 에이전트(agent.py)
  ↓
어떤 보안 도구를 사용할지 판단
  ↓
보안 도구들(tools.py)
  ├─ 파일 읽기
  ├─ 민감정보 검사
  ├─ 정적 보안 검사
  ├─ 보안 체크리스트
  └─ 위험도 계산
  ↓
분석 결과 설명
```

`agent.py`는 **이 AI가 어떤 역할을 하는가**를 정의하고, `tools.py`는 **실제로 어떤 보안 작업을 할 수 있는가**를 구현합니다. `langgraph.json`은 LangGraph Studio가 이 에이전트를 실행할 수 있게 연결합니다.


## 4. 기존 코딩 에이전트에서 바뀐 점

기존 코딩 에이전트는 파일 작성, 삭제, 디렉터리 생성, Python 코드 실행처럼 직접 작업을 수행하는 기능이 중심이었습니다.

현재 보안 분석 에이전트는 코드 수정이나 실행보다 **읽기와 분석**에 집중합니다. 파일을 확인하고, 위험 패턴을 탐지하고, 위험도와 대응 방법을 설명하는 구조입니다.

| 구분 | 기존 코딩 에이전트 | 현재 보안 분석 에이전트 |
|---|---|---|
| 목적 | 코드 작성/수정/실행 지원 | 소스코드/설정 파일 보안 1차 점검 |
| 주요 도구 | 파일 쓰기, 삭제, 실행 | 파일 읽기, 민감정보 탐지, 정적 보안 점검 |
| 응답 방식 | 작업 결과 중심 | 원인, 영향, 위험도, 대응 방법 중심 |
| 안전 원칙 | 작업 수행 보조 | 민감정보 노출 방지, 방어 중심 분석 |


In [ ]:
# agent.py의 핵심 내용을 확인합니다.
from pathlib import Path

agent_source = Path('agent.py').read_text(encoding='utf-8')
print(agent_source[:2000])


## 5. 보안 도구 목록 확인

`tools.py`에는 다음 도구가 구현되어 있습니다.

- `read_file`: 분석 대상 파일의 내용을 읽습니다.
- `list_directory`: 분석 가능한 파일과 폴더를 확인합니다.
- `scan_sensitive_information`: API Key, 비밀번호, JWT, Private Key 등 민감정보 의심 패턴을 탐지합니다.
- `static_security_scan`: `eval`, `exec`, `os.system`, `shell=True`, `pickle.load`, `verify=False` 같은 위험 패턴을 검사합니다.
- `security_checklist`: Web, API, Python, Container, Cloud 분야별 기본 점검 항목을 제공합니다.
- `calculate_risk_score`: 발생 가능성과 영향도를 기준으로 위험도를 계산합니다.


In [ ]:
from tools import SECURITY_TOOLS

print(f'등록된 보안 도구 수: {len(SECURITY_TOOLS)}개')

for index, tool in enumerate(SECURITY_TOOLS, start=1):
    print(f'{index}. {tool.name}')
    print(f'   {tool.description.splitlines()[0]}')


## 6. 파일 읽기/목록 확인 테스트

에이전트가 보안 분석을 하려면 먼저 어떤 파일이 있는지 확인하고, 필요한 파일을 읽을 수 있어야 합니다.


In [ ]:
from tools import list_directory, read_file

print(list_directory.invoke({'dir_path': '.'}))
print('\n' + '=' * 80 + '\n')
print(read_file.invoke({'file_path': 'langgraph.json'}))


## 7. 민감정보 탐지 테스트

아래 코드는 임시 파일을 만들어 민감정보 탐지 도구를 테스트합니다. 실제 API Key를 넣지 않고, 실행 중에만 만들어지는 데모 문자열을 사용합니다.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from tools import scan_sensitive_information

with TemporaryDirectory() as temp_dir:
    sample_path = Path(temp_dir) / 'sample_config.py'
    fake_key = 'sk-' + 'A' * 32
    sample_path.write_text(
        '\n'.join([
            '# 데모용 샘플 파일입니다. 실제 Secret이 아닙니다.',
            'password = "demo-only-password"',
            f'api_key = "{fake_key}"',
        ]),
        encoding='utf-8',
    )

    result = scan_sensitive_information.invoke({'file_path': str(sample_path)})
    print(result)


## 8. 정적 보안 점검 테스트

정적 보안 점검은 위험한 함수나 설정 패턴을 정규식 기반으로 찾습니다. 아래 예시는 임시 파일에 위험 패턴을 넣고 탐지 결과를 확인합니다.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from tools import static_security_scan

sample_code = '''
import os
import pickle
import requests

user_input = "1 + 1"
eval(user_input)
os.system("echo demo")
pickle.loads(b"demo")
requests.get("https://example.com", verify=False)
'''

with TemporaryDirectory() as temp_dir:
    sample_path = Path(temp_dir) / 'insecure_sample.py'
    sample_path.write_text(sample_code, encoding='utf-8')

    result = static_security_scan.invoke({'file_path': str(sample_path)})
    print(result)


## 9. 실제 프로젝트 파일 점검 예시

아래처럼 현재 프로젝트 파일을 대상으로도 점검할 수 있습니다. 결과가 없더라도, 이는 현재 정의된 정규식 규칙에 걸린 항목이 없다는 뜻이며 완전한 무취약을 의미하지는 않습니다.


In [ ]:
from tools import scan_sensitive_information, static_security_scan

print('[민감정보 점검]')
print(scan_sensitive_information.invoke({'file_path': 'tools.py'}))

print('\n' + '=' * 80 + '\n')

print('[정적 보안 점검]')
print(static_security_scan.invoke({'file_path': 'tools.py'}))


## 10. 보안 체크리스트 확인

분석 대상이 웹, API, Python, 컨테이너, 클라우드 중 어디에 가까운지에 따라 기본 점검 항목을 확인할 수 있습니다.


In [ ]:
from tools import security_checklist

for target_type in ['web', 'api', 'python', 'container', 'cloud']:
    print(security_checklist.invoke({'target_type': target_type}))
    print('\n' + '-' * 80 + '\n')


## 11. 위험도 계산 테스트

위험도는 발생 가능성(`likelihood`)과 영향도(`impact`)를 1~5점으로 입력해 계산합니다.

- 1~5점: Low
- 6~11점: Medium
- 12~19점: High
- 20~25점: Critical


In [ ]:
from tools import calculate_risk_score

examples = [
    {'likelihood': 1, 'impact': 3},
    {'likelihood': 3, 'impact': 4},
    {'likelihood': 5, 'impact': 5},
]

for example in examples:
    print(calculate_risk_score.invoke(example))
    print()


## 12. LangGraph Studio 실행 방법

루트 디렉터리에 `.env` 파일을 만들고 `OPENAI_API_KEY`를 설정한 뒤 실행합니다.

```bash
uv sync
uv run langgraph dev
```

Windows PowerShell에서는 다음 명령을 사용할 수 있습니다.

```powershell
$env:PYTHONUTF8=1; uv run langgraph dev --no-reload --allow-blocking
```

LangGraph Studio에서 다음과 같이 요청해볼 수 있습니다.

```text
tools.py 파일 보안 검사해줘
agent.py에 민감정보가 있는지 확인해줘
Python 보안 체크리스트 알려줘
발생 가능성 4, 영향도 5인 취약점의 위험도를 계산해줘
```


## 13. 결과 보고 형식

에이전트 응답은 아래 순서로 정리되도록 `agent.py`의 시스템 프롬프트에 설정되어 있습니다.

1. 요약
2. 발견 사항
3. 위험도
4. 대응 방법

또한 실제 비밀번호, API Key, 토큰 등 민감정보는 응답에 그대로 노출하지 않도록 안내되어 있습니다.


## 14. 한계점

현재 구현은 간단한 정규식 기반 정적 분석입니다. 따라서 다음 한계가 있습니다.

- 실제 데이터 흐름을 추적하지 못합니다.
- 사용자 입력이 위험 함수까지 도달하는지 정확히 판단하지 못합니다.
- 프레임워크별 보안 설정을 깊게 이해하지는 못합니다.
- 오탐과 누락이 발생할 수 있습니다.
- 전문 SAST 도구를 완전히 대체하지 않습니다.

따라서 이 프로젝트는 **취약점 완전 자동 탐지기**가 아니라, 보안 점검의 시작점을 빠르게 잡아주는 **1차 보안 분석 에이전트**로 설명하는 것이 정확합니다.


## 15. 앞으로 발전 방향

- 더 다양한 취약 패턴 추가: SQL Injection, XSS, 경로 조작, SSRF, CORS 오류 등으로 규칙을 확장합니다.
- 파일 확장자별 분석 전략 분리: Python, JavaScript, YAML, Dockerfile, `.env` 등 파일 종류에 맞게 점검합니다.
- 줄 번호와 코드 근거 강화: 탐지 결과를 더 읽기 쉽게 만들고 수정 예시까지 제공합니다.
- 위험도 계산 고도화: CVSS처럼 공격 난이도, 영향 범위, 노출 가능성을 세분화합니다.
- 전문 도구 연동: Bandit, Semgrep, Trivy 같은 보안 도구와 연결해 정확도를 높입니다.
- 보고서 자동 생성: Markdown, PDF, CSV 형태로 점검 결과를 저장합니다.
- CI/CD 연동: GitHub Actions에서 Pull Request마다 자동 보안 점검을 수행합니다.


## 16. 팀원 설명용 요약

기존 코딩 에이전트 구조를 보안 분석 에이전트로 변경했습니다.

사용자가 소스코드나 설정 파일의 보안 점검을 요청하면 에이전트가 파일을 확인하고, 민감정보 노출이나 위험한 코드 패턴을 자동으로 검사합니다.

검사 결과는 위험 원인, 영향, 위험도, 대응 방법 형태로 정리해서 제공합니다.

현재는 간단한 정규식 기반 정적 분석이라 전문 SAST 도구를 완전히 대체하는 건 아니고, 보안 점검의 1차 자동화 용도로 만든 구조입니다.

정확히는 **LLM + 커스텀 보안 도구를 연결한 1차 보안 분석 에이전트**입니다.
